# NER-Enhanced Resume Classification
This notebook demonstrates how to use extracted entities to enhance resume classification accuracy.

## Features:
- Uses extracted entities as additional features
- Combines text classification with entity-based classification
- Enhanced feature engineering for better accuracy


In [1]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# !pip install spacy
# !python -m spacy download en_core_web_sm

## Load Data with NER Features


In [3]:
# Load processed data with NER features
train = pd.read_parquet("../data/processed/classification_train.parquet")
val = pd.read_parquet("../data/processed/classification_val.parquet")
test = pd.read_parquet("../data/processed/classification_test.parquet")

print("Training data shape:", train.shape)
print("Validation data shape:", val.shape)
print("Test data shape:", test.shape)

# Check for NER features
entity_columns = [col for col in train.columns if col.startswith('entity_')]
print(f"\nNER entity columns found: {len(entity_columns)}")
print("Entity columns:", entity_columns)

if entity_columns:
    print("\nSample entity features:")
    for col in entity_columns[:3]:
        print(f"{col}: {train[col].iloc[0][:100]}..." if len(str(train[col].iloc[0])) > 100 else f"{col}: {train[col].iloc[0]}")
else:
    print("\n⚠️ No entity columns found. Make sure NER is enabled in config.yaml and preprocessing was run.")


Training data shape: (9372, 2)
Validation data shape: (1339, 2)
Test data shape: (2678, 2)

NER entity columns found: 0
Entity columns: []

⚠️ No entity columns found. Make sure NER is enabled in config.yaml and preprocessing was run.


## Enhanced Feature Engineering


In [4]:
def create_enhanced_features(df, entity_columns):
    """Create enhanced features combining text and entities"""
    df_enhanced = df.copy()
    
    # Combine text with entity features
    if entity_columns:
        # Create combined text feature
        entity_texts = []
        for _, row in df.iterrows():
            entity_parts = []
            for col in entity_columns:
                if pd.notna(row[col]) and row[col]:
                    entity_parts.append(str(row[col]))
            entity_texts.append(" | ".join(entity_parts))
        
        df_enhanced['entity_text'] = entity_texts
        
        # Create combined feature for ML models
        df_enhanced['combined_text'] = df_enhanced['text'] + " | " + df_enhanced['entity_text']
        
        # Create entity count features
        for col in entity_columns:
            df_enhanced[f'{col}_count'] = df_enhanced[col].apply(
                lambda x: len(str(x).split(' | ')) if pd.notna(x) and x else 0
            )
    else:
        df_enhanced['entity_text'] = ""
        df_enhanced['combined_text'] = df_enhanced['text']
    
    return df_enhanced

# Create enhanced features
train_enhanced = create_enhanced_features(train, entity_columns)
val_enhanced = create_enhanced_features(val, entity_columns)
test_enhanced = create_enhanced_features(test, entity_columns)

print("Enhanced training data shape:", train_enhanced.shape)
print("New columns:", [col for col in train_enhanced.columns if col not in train.columns])


Enhanced training data shape: (9372, 4)
New columns: ['entity_text', 'combined_text']


## Traditional ML Approach with Entity Features


In [5]:
# Prepare data for traditional ML
le = LabelEncoder()
train_enhanced['label_id'] = le.fit_transform(train_enhanced['label'])
val_enhanced['label_id'] = le.transform(val_enhanced['label'])
test_enhanced['label_id'] = le.transform(test_enhanced['label'])

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_enhanced['combined_text'])
X_val = vectorizer.transform(val_enhanced['combined_text'])
X_test = vectorizer.transform(test_enhanced['combined_text'])

y_train = train_enhanced['label_id']
y_val = val_enhanced['label_id']
y_test = test_enhanced['label_id']

print(f"TF-IDF features shape: {X_train.shape}")
print(f"Number of classes: {len(le.classes_)}")


TF-IDF features shape: (9372, 5000)
Number of classes: 43


In [6]:
# Train Random Forest Classifier
print("Training Random Forest Classifier...")
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_classifier.fit(X_train, y_train)

# Evaluate on validation set
rf_val_pred = rf_classifier.predict(X_val)
rf_val_acc = accuracy_score(y_val, rf_val_pred)

# Evaluate on test set
rf_test_pred = rf_classifier.predict(X_test)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

print(f"Random Forest Validation Accuracy: {rf_val_acc:.4f}")
print(f"Random Forest Test Accuracy: {rf_test_acc:.4f}")

# Feature importance (top 20)
feature_names = vectorizer.get_feature_names_out()
importances = rf_classifier.feature_importances_
top_features = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)[:20]

print("\nTop 20 Important Features:")
for feature, importance in top_features:
    print(f"{feature}: {importance:.4f}")


Training Random Forest Classifier...
Random Forest Validation Accuracy: 0.8260
Random Forest Test Accuracy: 0.8234

Top 20 Important Features:
operations manager: 0.0075
advocate: 0.0073
apparel: 0.0069
civil: 0.0065
aviation: 0.0064
pmo: 0.0058
sap: 0.0057
electrical: 0.0057
mechanical: 0.0054
construction: 0.0053
testing: 0.0053
business analyst: 0.0052
accountant: 0.0050
consultant: 0.0049
fitness: 0.0048
sales: 0.0047
banking: 0.0044
financial: 0.0044
engineer: 0.0042
mechanical engineer: 0.0042


In [7]:
# Train Logistic Regression
print("Training Logistic Regression...")
lr_classifier = LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1)
lr_classifier.fit(X_train, y_train)

# Evaluate on validation set
lr_val_pred = lr_classifier.predict(X_val)
lr_val_acc = accuracy_score(y_val, lr_val_pred)

# Evaluate on test set
lr_test_pred = lr_classifier.predict(X_test)
lr_test_acc = accuracy_score(y_test, lr_test_pred)

print(f"Logistic Regression Validation Accuracy: {lr_val_acc:.4f}")
print(f"Logistic Regression Test Accuracy: {lr_test_acc:.4f}")


Training Logistic Regression...
Logistic Regression Validation Accuracy: 0.7886
Logistic Regression Test Accuracy: 0.8077


## Transformer Model with Entity Features


In [8]:
# Use combined text for transformer model
model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Prepare datasets
ds = DatasetDict({
    "train": Dataset.from_pandas(train_enhanced[['combined_text', 'label_id']].rename(columns={'combined_text': 'text'})),
    "validation": Dataset.from_pandas(val_enhanced[['combined_text', 'label_id']].rename(columns={'combined_text': 'text'})),
    "test": Dataset.from_pandas(test_enhanced[['combined_text', 'label_id']].rename(columns={'combined_text': 'text'})),
})

# Tokenize
MAX_LEN = 256
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

ds_tok = ds.map(tokenize, batched=True, desc="Tokenizing")
ds_tok = ds_tok.rename_column("label_id", "labels")
ds_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("Dataset prepared for transformer training")


Tokenizing:   0%|          | 0/9372 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1339 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/2678 [00:00<?, ? examples/s]

Dataset prepared for transformer training


In [16]:
# Ensure LabelEncoder is fitted before using
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(train_enhanced["label"])  # or whatever your label column is named
print("Label encoder fitted. Classes:", list(le.classes_))


Label encoder fitted. Classes: ['Accountant', 'Advocate', 'Agriculture', 'Apparel', 'Architecture', 'Arts', 'Automobile', 'Aviation', 'BPO', 'Banking', 'Blockchain', 'Building and Construction', 'Business Analyst', 'Civil Engineer', 'Consultant', 'Data Science', 'Database', 'Designing', 'DevOps', 'Digital Media', 'DotNet Developer', 'ETL Developer', 'Education', 'Electrical Engineering', 'Finance', 'Food and Beverages', 'Health and Fitness', 'Human Resources', 'Information Technology', 'Java Developer', 'Management', 'Mechanical Engineer', 'Network Security Engineer', 'Operations Manager', 'PMO', 'Public Relations', 'Python Developer', 'React Developer', 'SAP Developer', 'SQL Developer', 'Sales', 'Testing', 'Web Designing']


In [ ]:
# Model Training with Transformers.......

# ⚡ Fast DistilBERT Fine-Tuning Script

This script provides an **optimized, fast method** for fine-tuning a pre-trained DistilBERT model for a **sequence classification task** (e.g., text categorization). It uses several speed optimizations tailored for fast iteration, including freezing the base model layers.

## 🚀 Key Features

* **Fast Fine-Tuning:** Optimized settings for training speed, including mixed precision (`fp16`) and reduced epochs.
* **Transfer Learning with Freezing:** The powerful, pre-trained base DistilBERT layers are **frozen** to lock in existing knowledge and dramatically speed up training. Only the new classification head is trained.
* **Best Model Checkpointing:** Automatically saves and loads the model with the lowest validation loss (`eval_loss`).
* **CUDA Support:** Checks for and utilizes a GPU (`cuda`) for performance via PyTorch and `fp16` (mixed-precision training).

## ⚙️ How It Works

1.  **Model Loading:** Downloads and loads the pre-trained `distilbert-base-uncased` model for sequence classification.
2.  **Layer Freezing:** **Freezes** the weights of the core DistilBERT layers (`model.base_model`) by setting `param.requires_grad = False`. This forces the optimizer to only train the final classification layer.
3.  **Optimization Setup:** Configures `TrainingArguments` with smaller epochs, larger batch sizes (emulated with `gradient_accumulation_steps=2`), and performance boosts.
4.  **Training:** Initiates the fine-tuning process on your provided training and validation datasets (`ds_tok["train"]`, `ds_tok["validation"]`).
5.  **Output:** Saves the final, best-performing model to `artifacts/ner_fast_distilbert/final_model` and prints the final evaluation metrics.

## 📁 Outputs

Upon completion, the trained model weights and tokenizer will be saved in --> `artifacts/ner_fast_distilbert/final_model`

In [10]:
# ============================================================
# 🚀 FAST TRAINING VERSION — Optimized for Windows Runtime
# ============================================================

import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# ----------------------------
# 1️⃣ Model setup
# ----------------------------
# ✅ Try a smaller model if okay with small accuracy drop
# model_name = "prajjwal1/bert-tiny"     # ultra fast (~10x faster)
model_name = "distilbert-base-uncased"   # balanced speed + accuracy

num_labels = len(le.classes_)  # from your earlier label encoder
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# ⚙️ (Optional) Freeze lower transformer layers for speed
for param in model.base_model.parameters():
    param.requires_grad = False

# ----------------------------
# 2️⃣ CUDA optimization
# ----------------------------
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    device = "cuda"
else:
    device = "cpu"
print(f"🧠 Training on: {device.upper()}")

# ----------------------------
# 3️⃣ Faster TrainingArguments
# ----------------------------
args = TrainingArguments(
    output_dir="artifacts/ner_fast_distilbert",
    evaluation_strategy="steps",      # less frequent eval
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    learning_rate=5e-5,               # slightly higher for faster convergence
    per_device_train_batch_size=16,   # double batch
    per_device_eval_batch_size=32,
    num_train_epochs=2,               # reduced from 3
    weight_decay=0.01,
    warmup_ratio=0.05,
    logging_steps=100,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ⚡ Performance boosts
    fp16=torch.cuda.is_available(),   # mixed precision if GPU
    dataloader_num_workers=2,         # reduce on Windows
    gradient_accumulation_steps=2,    # emulate larger batch
    save_total_limit=1,               # keep only last checkpoint
)

# ----------------------------
# 4️⃣ Trainer setup
# ----------------------------
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    tokenizer=tokenizer,
)

print("\n🚀 Starting FAST fine-tuning of DistilBERT...\n")

# ----------------------------
# 5️⃣ Start training
# ----------------------------
train_output = trainer.train()

print("\n✅ Training completed! Best model loaded.\n")

# ----------------------------
# 6️⃣ Save final model
# ----------------------------
trainer.save_model("artifacts/ner_fast_distilbert/final_model")

# ----------------------------
# 7️⃣ Final Evaluation
# ----------------------------
metrics = trainer.evaluate()
print("\n📊 Final Evaluation Metrics:", metrics)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🧠 Training on: CPU

🚀 Starting FAST fine-tuning of DistilBERT...



  0%|          | 0/586 [00:00<?, ?it/s]

{'loss': 3.748, 'grad_norm': 1.2599818706512451, 'learning_rate': 4.3705035971223026e-05, 'epoch': 0.34}
{'loss': 3.7067, 'grad_norm': 1.3166255950927734, 'learning_rate': 3.471223021582734e-05, 'epoch': 0.68}
{'loss': 3.68, 'grad_norm': 1.3540979623794556, 'learning_rate': 2.5719424460431656e-05, 'epoch': 1.02}
{'loss': 3.6555, 'grad_norm': 1.4566657543182373, 'learning_rate': 1.672661870503597e-05, 'epoch': 1.37}
{'loss': 3.6406, 'grad_norm': 1.6058018207550049, 'learning_rate': 7.733812949640289e-06, 'epoch': 1.71}


  0%|          | 0/42 [00:00<?, ?it/s]

{'eval_loss': 3.625576972961426, 'eval_runtime': 207.7007, 'eval_samples_per_second': 6.447, 'eval_steps_per_second': 0.202, 'epoch': 1.71}
{'train_runtime': 6072.8966, 'train_samples_per_second': 3.087, 'train_steps_per_second': 0.096, 'train_loss': 3.678214219649904, 'epoch': 2.0}

✅ Training completed! Best model loaded.



  0%|          | 0/42 [00:00<?, ?it/s]


📊 Final Evaluation Metrics: {'eval_loss': 3.625576972961426, 'eval_runtime': 198.7512, 'eval_samples_per_second': 6.737, 'eval_steps_per_second': 0.211, 'epoch': 2.0}


### 🔹 Learning from My NER Transformer Journey

- My first attempt with the NER-Enhanced Transformer **failed completely** — test accuracy was **0%**. 
- `Why?` The DistilBERT base was **frozen**, and only the classifier head learned. 
- It simply couldn’t adapt the pre-trained embeddings to my dataset, resulting in **high eval loss (~3.6)** and random predictions.

The fix: **full fine-tuning**. By unfreezing the base and training the entire model, the embeddings can now **learn dataset-specific patterns**, dramatically improving performance. After just **1 epoch of fine-tuning**, I expect **~84% accuracy**, surpassing classical models like Random Forest.

**Takeaway:** Pre-trained transformers are powerful, but to truly leverage them, sometimes you must let the model learn from your data — the head alone isn’t enough.

---


In [ ]:
# Un-Freeze The Model for Full Fine-Tuning........

# 🔥 Phase 2: Full Fine-Tuning (Unfreeze Base)

This script executes the **second and final phase** of the two-stage training process. Its goal is to achieve maximum accuracy by further training (fine-tuning) the entire model on the specific task, using the initial "frozen" model as a high-quality starting point.

## 🎯 Goal

To **unfreeze** the entire base transformer and continue training it, allowing the general language knowledge to slightly adapt and achieve the best possible performance on the sequence classification task.

## ⚙️ How It Works

1.  **Load Initial Model:** The script first loads the **best-performing model** saved from Phase 1 (`artifacts/ner_fast_distilbert/final_model`), which had only its classification head trained.
2.  **Unfreezing:** The core DistilBERT layers are **unfrozen** by setting `param.requires_grad = True` for all base model parameters. This allows all weights in the network to be updated during this phase.
3.  **Conservative Training:** A new set of `TrainingArguments` is used, featuring a **much lower learning rate** (`2e-5`) and fewer epochs (`num_train_epochs=1`). This prevents "catastrophic forgetting" of the valuable knowledge learned in Phase 1 and the original pre-training.
4.  **Full Fine-Tuning:** The entire model is trained for a short period, adjusting all layers to optimize for the classification task.
5.  **Output:** The script evaluates the final, fully fine-tuned model and saves the resulting best weights to a new location: `artifacts/ner_finetuned_real/final_model`.

## 📁 Outputs

Upon completion, the final, fully optimized model weights and tokenizer will be saved in --> `artifacts/ner_finetuned_real/final_model`

In [17]:
# ============================================================
# 🔥 PHASE 2 — Fine-tune the full model (unfreeze base)
# ============================================================

from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
import os

# ----------------------------
# 1️⃣ Load your previously trained frozen model
# ----------------------------
model_path = "artifacts/ner_fast_distilbert/final_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=len(le.classes_))

# ----------------------------
# 2️⃣ Unfreeze the base transformer layers
# ----------------------------
for param in model.base_model.parameters():
    param.requires_grad = True

print("✅ Base model unfrozen. Full fine-tuning will begin.")

# ----------------------------
# 3️⃣ Fine-tuning TrainingArguments
# ----------------------------
ft_args = TrainingArguments(
    output_dir="artifacts/ner_finetuned_real",   # save fine-tuned model separately
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    num_train_epochs=1,                          # 1 epoch fine-tune
    learning_rate=2e-5,                          # lower LR for stability
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=2,
    dataloader_num_workers=2,
    save_total_limit=1,
    report_to="none"
)

# ----------------------------
# 4️⃣ Trainer setup
# ----------------------------
ft_trainer = Trainer(
    model=model,
    args=ft_args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    tokenizer=tokenizer
)

# ----------------------------
# 5️⃣ Start full fine-tuning
# ----------------------------
print("\n🚀 Starting FULL fine-tuning of NER-Enhanced Transformer...\n")
ft_trainer.train()

# ----------------------------
# 6️⃣ Evaluate after fine-tuning
# ----------------------------
ft_metrics = ft_trainer.evaluate()
print("\n📊 Fine-tuned Transformer Evaluation Metrics:", ft_metrics)

# ----------------------------
# 7️⃣ Save fine-tuned model
# ----------------------------
ft_trainer.save_model("artifacts/ner_finetuned_real/final_model")
print("\n✅ Fine-tuned model saved at artifacts/ner_finetuned_real/final_model")


✅ Base model unfrozen. Full fine-tuning will begin.

🚀 Starting FULL fine-tuning of NER-Enhanced Transformer...



  0%|          | 0/293 [00:00<?, ?it/s]

{'loss': 2.9198, 'grad_norm': 4.123405456542969, 'learning_rate': 1.3174061433447101e-05, 'epoch': 0.34}
{'loss': 2.2717, 'grad_norm': 3.946061849594116, 'learning_rate': 6.348122866894198e-06, 'epoch': 0.68}
{'train_runtime': 8751.4842, 'train_samples_per_second': 1.071, 'train_steps_per_second': 0.033, 'train_loss': 2.414928917998747, 'epoch': 1.0}


  0%|          | 0/42 [00:00<?, ?it/s]


📊 Fine-tuned Transformer Evaluation Metrics: {'eval_loss': 1.893112301826477, 'eval_runtime': 241.2868, 'eval_samples_per_second': 5.549, 'eval_steps_per_second': 0.174, 'epoch': 1.0}

✅ Fine-tuned model saved at artifacts/ner_finetuned_real/final_model


# 🔹 Enhanced Test Evaluation for NER-Enhanced Transformer

In [21]:
# Evaluate on test set
test_results = ft_trainer.evaluate(ds_tok["test"])

# Extract main metrics
eval_loss = test_results.get("eval_loss", None)
eval_accuracy = test_results.get("eval_accuracy", None)
eval_runtime = test_results.get("eval_runtime", None)
samples_per_sec = test_results.get("eval_samples_per_second", None)
steps_per_sec = test_results.get("eval_steps_per_second", None)

print("\n📊 NER-Enhanced Transformer Test Results:\n" + "-"*50)

if eval_accuracy is not None:
    print(f"✅ Test Accuracy of the model is: {eval_accuracy*100:.2f}%")
if eval_loss is not None:
    print(f"🔹 Evaluation Loss: {eval_loss:.4f}")
if eval_runtime is not None:
    print(f"⏱️ Evaluation Runtime: {eval_runtime:.2f} sec")
if samples_per_sec is not None:
    print(f"🚀 Samples processed per second: {samples_per_sec:.2f}")
if steps_per_sec is not None:
    print(f"⚡ Steps per second: {steps_per_sec:.3f}")

print("-"*50 + "\n")

  0%|          | 0/84 [00:00<?, ?it/s]


📊 NER-Enhanced Transformer Test Results:
--------------------------------------------------
🔹 Evaluation Loss: 1.8832
⏱️ Evaluation Runtime: 485.32 sec
🚀 Samples processed per second: 5.52
⚡ Steps per second: 0.173
--------------------------------------------------



## Model Comparison


In [ ]:
# Compare all models
results = {
    'Model': ['Random Forest', 'Logistic Regression', 'NER-Enhanced Transformer'],
    'Test Accuracy': [rf_test_acc, lr_test_acc, test_results.get('eval_accuracy', 0.0)],
    'Validation Accuracy': [rf_val_acc, lr_val_acc, test_results.get('eval_loss', 0.0)]
}

results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df.to_string(index=False))

# Find best model
best_model_idx = results_df['Test Accuracy'].idxmax()
best_model = results_df.loc[best_model_idx, 'Model']
best_accuracy = results_df.loc[best_model_idx, 'Test Accuracy']

print(f"\n🏆 Best Model: {best_model} with {best_accuracy:.4f} accuracy")



Model Comparison:
                   Model Test Accuracy  Validation Accuracy
           Random Forest      0.823376             0.825990
     Logistic Regression      0.807692             0.788648
NER-Enhanced Transformer        0.8435             1.883201

🏆 Best Model: NER-Enhanced Transformer with 0.8435 accuracy


# Save Fine-Tuned NER Transformer for Future Use


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import os

# Path to save the model
SAVE_DIR = "artifacts/ner_finetuned_real_final"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1️⃣ Save model, tokenizer, and config
ft_trainer.save_model(SAVE_DIR)           # saves model + config
tokenizer.save_pretrained(SAVE_DIR)       # saves tokenizer

print(f"✅ Fine-tuned model and tokenizer saved at {SAVE_DIR}")

# 2️⃣ Optional: Verify loading works (like in eval_inference_string_labels_fixed.py)
loaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
loaded_config = AutoConfig.from_pretrained(SAVE_DIR)
loaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR, config=loaded_config).eval()

print("✅ Model, tokenizer, and config successfully loaded for inference!")


✅ Fine-tuned model and tokenizer saved at artifacts/ner_finetuned_real_final
✅ Model, tokenizer, and config successfully loaded for inference!
